# Amazon Electronics: exploration and preparation

- Inspect source records and prepare product text with CARec.
- Sample interactions, create the warm/cold split, then save and verify the data.
- The saved outputs below show the completed preparation.
- Load the results for encoder and recommender experiments in [carec_encoder_comparison.ipynb](carec_encoder_comparison.ipynb).


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


## Paths and preview

- Find the project root from the repository or `src/`.
- Read a few initial JSONL records. Each line contains one record.
- Use the saved sample when the raw files have been removed.
- These first records describe only the file beginnings. They are not a random sample or a full data-quality check.


In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError(
        "Select the 'NI-PML benchmarks' kernel or .venv/bin/python, then restart and run the notebook. "
        f"Current Python: {sys.executable}"
    )

DATA_DIR = PROJECT_ROOT / "data"
REVIEWS_PATH = DATA_DIR / "Electronics.json"
METADATA_PATH = DATA_DIR / "meta_Electronics.json"

PREPARED_DIR = DATA_DIR / "processed/electronics_carec_035_seed2023"
USE_RAW_PREVIEW = REVIEWS_PATH.is_file() and METADATA_PATH.is_file()
if USE_RAW_PREVIEW:
    for path in (REVIEWS_PATH, METADATA_PATH):
        print(f"{path.relative_to(PROJECT_ROOT)}")
elif (PREPARED_DIR / "manifest.json").is_file():
    print(f"Using saved data from {PREPARED_DIR}")
else:
    raise FileNotFoundError("Neither both raw datasets nor a prepared dataset are available.")

data/Electronics.json
data/meta_Electronics.json


### Data preview

- Read only the requested number of lines.
- Report the file and line number if JSON parsing fails.


In [4]:
import gzip
import json
from itertools import islice

import pandas as pd
from IPython.display import display


def read_jsonl_preview(path, limit=5):
    if limit < 1:
        raise ValueError("Preview limit must be positive.")
    records = []
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as source:
        for line_number, line in enumerate(islice(source, limit), start=1):
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON in {path}, line {line_number}") from error
    return records

In [5]:
if USE_RAW_PREVIEW:
    review_preview = read_jsonl_preview(REVIEWS_PATH, 2)
    metadata_preview = read_jsonl_preview(METADATA_PATH, 2)
else:
    review_preview = pd.read_csv(
        PREPARED_DIR / "sampled_interactions.tsv.gz", sep="\t", nrows=2,
        dtype={"user_id:token": str, "item_id:token": str},
    ).rename(columns={
        "user_id:token": "reviewerID", "item_id:token": "asin",
        "raings:float": "overall", "timestamp:float": "unixReviewTime",
    }).to_dict("records")
    metadata_preview = read_jsonl_preview(PREPARED_DIR / "sampled_metadata.jsonl.gz", 2)
    metadata_preview = [
        {("asin" if key == "item_id:token" else key): value for key, value in item.items()}
        for item in metadata_preview
    ]
print(f"Reviews: first {len(review_preview)} records")
display(pd.DataFrame(review_preview))
print(f"Metadata: first {len(metadata_preview)} records")
display(pd.DataFrame(metadata_preview))


Reviews: first 2 records


,overall,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime
0,5.0,True,"07 17, 2002",A1N070NS9CJQ2I,0060009810,{'Format:': ' Hardcover'},Teri Adams,This was the first time I read Garcia-Aguilera...,Hit The Spot!,1026864000
1,5.0,False,"07 6, 2002",A3P0KRKOBQK1KN,0060009810,{'Format:': ' Hardcover'},Willa C.,"As with all of Ms. Garcia-Aguilera's books, I ...",one hot summer is HOT HOT HOT!,1025913600


Metadata: first 2 records


,category,tech1,description,fit,title,also_buy,tech2,brand,feature,rank,also_view,main_cat,similar_item,date,price,asin,imageURL,imageURLHighRes
0,"[Electronics, Camera &amp; Photo, Video Survei...",,[The following camera brands and models have b...,,Genuine Geovision 1 Channel 3rd Party NVR IP S...,[],,GeoVision,"[Genuine Geovision 1 Channel NVR IP Software, ...","[>#3,092 in Tools &amp; Home Improvement &gt; ...",[],Camera &amp; Photo,,"January 28, 2014",$65.00,0011300000,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...
1,"[Electronics, Camera &amp; Photo]",,[This second edition of the Handbook of Astron...,,"Books ""Handbook of Astronomical Image Processi...",[0999470906],,33 Books Co.,[Detailed chapters cover these fundamental top...,"[>#55,933 in Camera &amp; Photo (See Top 100 i...","[0943396670, 1138055360, 0999470906]",Camera &amp; Photo,,"June 17, 2003",,0043396828,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...


## Original CARec product text

- Load the pinned CARec source and Electronics YAML.
- Call `RawDataProcessor._preprocess_text_feature` on the preview.
- Combine title, brand, and category with the original cleaning rules.
- Use the same method later for the saved catalogue text.
- This step prepares text strings. Encoder runs belong in the experiment notebook.


In [6]:
import importlib.util
import sys

import yaml

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmarks.carec import load_carec

carec = load_carec()
spec = importlib.util.spec_from_file_location("carec_data_utils", carec.root / "data/utils.py")
carec_data = importlib.util.module_from_spec(spec)
spec.loader.exec_module(carec_data)

text_config = yaml.safe_load(
    (carec.root / "data/data_process_config_amazon_Electronics.yaml").read_text()
)
# Create the text processor without the constructor's output-directory creation.
text_processor = carec_data.RawDataProcessor.__new__(carec_data.RawDataProcessor)
text_processor.data_name = text_config["data_name"]
text_processor.text_feature_list = text_config["text_feature_list"]

product_text_frame = text_processor._preprocess_text_feature(
    pd.DataFrame(metadata_preview).copy(), "amazon"
)
product_text_preview = product_text_frame[["asin", "text"]].to_dict("records")
for item in product_text_preview:
    print(f"{item['asin']}\n{item['text']!r}\n")


/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0011300000
'Genuine Geovision 1 Channel 3rd Party NVR IP Software with USB Dongle Onvif PSIA GeoVision  Camera & Photo Video Surveillance Surveillance Systems Surveillance DVR Kits.'

0043396828
'Books Handbook of Astronomical Image Processing with CD ROM, 2nd Edition, Hardcover Book by Berry & Burnell 33 Books Co.  Camera & Photo.'



## Prepare and save the CARec sample

- Sample 35% of review rows with seed `2023`, preserving pandas sampling order.
- Apply user/item 5-core filtering, then match products to metadata.
- Hold out 5% of products as cold items.
- Repeat warm 5-core filtering and keep cold interactions for users remaining in warm data.
- Randomly split each user's warm history into train, validation, and test at 80/10/10.
- Use the original CARec filtering and splitting methods. Stream only the required raw fields.
- Save the initial sample, matching metadata, final partitions, and original CARec product texts.
- Verify reload and reconstruction before removing the raw files.
- Selecting a different sample from the full population requires the original data again.


In [7]:
import gzip
import hashlib
import time

PREPARED_DIR = DATA_DIR / "processed/electronics_carec_035_seed2023"
MANIFEST_PATH = PREPARED_DIR / "manifest.json"
BUILD_DATA = not MANIFEST_PATH.exists()
SAMPLE_FRACTION = text_config["inter_sample_ratio"]
SPLIT_SEED = text_config["seed"]


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_saved_interactions(path):
    return pd.read_csv(
        path, sep="\t", dtype={"user_id:token": str, "item_id:token": str},
        float_precision="round_trip",
    )


def read_saved_metadata(path):
    with gzip.open(path, "rt", encoding="utf-8") as source:
        return pd.DataFrame(json.loads(line) for line in source)


print(f"Sample fraction: {SAMPLE_FRACTION}; seed: {SPLIT_SEED}")
print(f"Output: {PREPARED_DIR}")
print("Build from raw files" if BUILD_DATA else "Reload the existing prepared dataset")


Sample fraction: 0.35; seed: 2023
Output: /Users/matouskovar/FIT/NI-PML/data/processed/electronics_carec_035_seed2023
Reload the existing prepared dataset


In [8]:
if BUILD_DATA:
    if PREPARED_DIR.exists():
        raise FileExistsError(f"Incomplete output already exists: {PREPARED_DIR}. Inspect it before rebuilding.")
    for path in (REVIEWS_PATH, METADATA_PATH):
        if not path.is_file():
            raise FileNotFoundError(path)
    preparation_started = time.monotonic()
    source_fingerprints = {}
    digest = hashlib.sha256()
    raw_review_rows = 0
    last_byte = b""
    review_stat = REVIEWS_PATH.stat()
    with REVIEWS_PATH.open("rb") as source:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
            raw_review_rows += chunk.count(b"\n")
            last_byte = chunk[-1:]
    if last_byte and last_byte != b"\n":
        raw_review_rows += 1
    source_fingerprints["reviews"] = {
        "file": REVIEWS_PATH.name, "bytes": review_stat.st_size,
        "mtime_ns": review_stat.st_mtime_ns, "rows": raw_review_rows,
        "sha256": digest.hexdigest(),
    }
    selected_rows = np.random.RandomState(SPLIT_SEED).choice(
        raw_review_rows, size=round(raw_review_rows * SAMPLE_FRACTION), replace=False
    )
    selected_mask = np.zeros(raw_review_rows, dtype=bool)
    selected_mask[selected_rows] = True
    print(f"Raw reviews: {raw_review_rows:,}; selected: {len(selected_rows):,}")
else:
    saved_manifest = json.loads(MANIFEST_PATH.read_text())
    assert saved_manifest["upstream_commit"] == carec.commit
    assert saved_manifest["sample_fraction"] == SAMPLE_FRACTION
    assert saved_manifest["seed"] == SPLIT_SEED
    source_fingerprints = saved_manifest["sources"]
    print("Using the saved source fingerprints and sample.")


Using the saved source fingerprints and sample.


In [9]:
if BUILD_DATA:
    review_records = []
    source_rows = []
    review_fields = list(text_config["inter_data_col"])
    observed_rows = 0
    with REVIEWS_PATH.open("rb") as source:
        for row_number, line in enumerate(source):
            observed_rows = row_number + 1
            if row_number >= raw_review_rows:
                raise RuntimeError("The raw review file changed during preparation.")
            if not selected_mask[row_number]:
                continue
            try:
                record = json.loads(line)
                values = [record[field] for field in review_fields]
                if not all(isinstance(value, str) and value for value in values[:2]):
                    raise ValueError("Invalid user or product ID")
                if not all(isinstance(value, (int, float)) and np.isfinite(value) for value in values[2:]):
                    raise ValueError("Invalid rating or timestamp")
            except (ValueError, KeyError, TypeError) as error:
                raise ValueError(f"Invalid selected review at line {row_number + 1}") from error
            review_records.append(values)
            source_rows.append(row_number)
            if len(review_records) % 1_000_000 == 0:
                print(f"Loaded {len(review_records):,} selected reviews")
    assert observed_rows == raw_review_rows
    current_stat = REVIEWS_PATH.stat()
    assert (current_stat.st_size, current_stat.st_mtime_ns) == (review_stat.st_size, review_stat.st_mtime_ns)
    sampled_reviews = pd.DataFrame(
        review_records, index=source_rows, columns=text_config["inter_data_col"].values()
    ).loc[selected_rows].copy()
    sampled_reviews.index.name = "source_row"
    sampled_reviews = sampled_reviews.reset_index()
    del review_records, source_rows, selected_mask, selected_rows
else:
    sampled_reviews = read_saved_interactions(PREPARED_DIR / "sampled_interactions.tsv.gz")
print(f"Selected interaction rows: {len(sampled_reviews):,}")


Selected interaction rows: 7,348,024


In [10]:
if BUILD_DATA:
    sampled_item_ids = set(sampled_reviews["item_id:token"])
    metadata_records = []
    seen_item_ids = set()
    duplicate_metadata_ids = 0
    metadata_rows = 0
    metadata_stat = METADATA_PATH.stat()
    digest = hashlib.sha256()
    with METADATA_PATH.open("rb") as source:
        for row_number, line in enumerate(source):
            metadata_rows = row_number + 1
            digest.update(line)
            try:
                record = json.loads(line)
                item_id = record.get("asin")
            except (ValueError, AttributeError) as error:
                raise ValueError(f"Invalid metadata at line {row_number + 1}") from error
            if item_id not in sampled_item_ids:
                continue
            if item_id in seen_item_ids:
                duplicate_metadata_ids += 1
                continue
            retained = {field: record[field] for field in text_config["item_meta_data_col"]}
            retained["metadata_source_row"] = row_number
            metadata_records.append(retained)
            seen_item_ids.add(item_id)
    current_stat = METADATA_PATH.stat()
    assert (current_stat.st_size, current_stat.st_mtime_ns) == (metadata_stat.st_size, metadata_stat.st_mtime_ns)
    source_fingerprints["metadata"] = {
        "file": METADATA_PATH.name, "bytes": metadata_stat.st_size,
        "mtime_ns": metadata_stat.st_mtime_ns, "rows": metadata_rows,
        "sha256": digest.hexdigest(), "duplicate_sampled_ids_skipped": duplicate_metadata_ids,
    }
    sampled_metadata = pd.DataFrame(metadata_records).rename(columns=text_config["item_meta_data_col"])
    del metadata_records, seen_item_ids, sampled_item_ids
else:
    sampled_metadata = read_saved_metadata(PREPARED_DIR / "sampled_metadata.jsonl.gz")
print(f"Metadata available for {len(sampled_metadata):,} sampled products")


Metadata available for 519,565 sampled products


### Apply the original CARec methods

- Return the already selected rows when CARec requests its initial sample.
- Check the fraction and seed so the 35% subset is not sampled twice.
- Run the unchanged `_preprocess_inter_data` method for filtering and cold selection.
- Preserve repeated user/product interactions, as upstream does.


In [11]:
class SelectedReviewSource:
    def __init__(self, frame):
        self.frame = frame

    def sample(self, frac, random_state):
        if (frac, random_state) != (SAMPLE_FRACTION, SPLIT_SEED):
            raise ValueError("CARec requested settings different from our saved sample.")
        return self.frame.copy()


def prepare_with_carec(reviews, metadata):
    processor = carec_data.RawDataProcessor.__new__(carec_data.RawDataProcessor)
    for name in (
        "data_name", "text_feature_list", "inter_sample_ratio", "seed",
        "user_k_core_threshold", "item_k_core_threshold", "cold_ratio", "eval_args",
    ):
        setattr(processor, name, text_config[name])
    processor.inter_data = SelectedReviewSource(reviews)
    processor.item_meta_data = metadata.copy()
    processor._preprocess_inter_data()
    return processor


processor = prepare_with_carec(sampled_reviews, sampled_metadata)
print(f"Warm rows: {len(processor.inter_data):,}; cold rows: {len(processor.cold_inter_data):,}")


    Epoch 1 The number of inters: 1139462, users: 4765287, items: 519866
    Epoch 2 The number of inters: 893943, users: 161153, items: 134139
    Epoch 3 The number of inters: 751231, users: 138970, items: 47301
    Epoch 4 The number of inters: 719346, users: 101917, items: 44139
    Epoch 5 The number of inters: 695647, users: 99380, items: 37960
    Epoch 6 The number of inters: 689623, users: 93783, items: 37491
    Epoch 7 The number of inters: 685019, users: 93346, items: 36392
    Epoch 8 The number of inters: 683986, users: 92267, items: 36313
    Epoch 9 The number of inters: 683182, users: 92198, items: 36122
    Epoch 10 The number of inters: 683015, users: 92004, items: 36115
    Epoch 11 The number of inters: 682880, users: 91994, items: 36083
    Epoch 12 The number of inters: 682832, users: 91962, items: 36081
    Epoch 13 The number of inters: 682796, users: 91959, items: 36072
    Epoch 14 The number of inters: 682784, users: 91950, items: 36072
    Epoch 15 The numb

In [12]:
train, valid, test = carec_data.split_data(
    processor.inter_data, [0.8, 0.1, 0.1], SPLIT_SEED, "user_id:token"
)
partitions = {"train": train, "valid": valid, "test": test, "cold": processor.cold_inter_data}
catalogue_metadata = processor.item_meta_data.copy()
original_text = processor._preprocess_text_feature(catalogue_metadata.copy(), "amazon")
items = catalogue_metadata.copy()
items["text"] = original_text["text"]
counts = {
    name: {"rows": len(frame), "users": int(frame["user_id:token"].nunique()),
           "products": int(frame["item_id:token"].nunique())}
    for name, frame in partitions.items()
}
print(pd.DataFrame(counts).T.to_string())
print(f"Catalogue: {len(items):,} products; empty upstream texts: {(items['text'] == '').sum():,}")


Split data:: 100%|██████████| 81520/81520 [00:12<00:00, 6389.18it/s] 


         rows  users  products
train  452606  81520     32416
valid   14853  12386      9322
test   133274  81520     28515
cold    23163  18680      1797
Catalogue: 34,220 products; empty upstream texts: 1


In [13]:
if BUILD_DATA:
    PREPARED_DIR.mkdir(parents=True, exist_ok=False)
    compression = {"method": "gzip", "compresslevel": 1}
    sampled_reviews.to_csv(PREPARED_DIR / "sampled_interactions.tsv.gz", sep="\t", index=False, compression=compression)
    sampled_metadata.to_json(PREPARED_DIR / "sampled_metadata.jsonl.gz", orient="records", lines=True, force_ascii=False, compression=compression)
    for name, frame in partitions.items():
        frame.to_csv(PREPARED_DIR / f"{name}.tsv.gz", sep="\t", index=False, compression=compression)
    items.to_json(PREPARED_DIR / "items.jsonl.gz", orient="records", lines=True, force_ascii=False, compression=compression)
    user_ids = sorted(processor.inter_data["user_id:token"].unique())
    item_ids = items["item_id:token"].tolist()
    pd.DataFrame({"user_id": user_ids, "index": range(len(user_ids))}).to_csv(PREPARED_DIR / "user_ids.tsv.gz", sep="\t", index=False, compression=compression)
    pd.DataFrame({"item_id": item_ids, "index": range(len(item_ids))}).to_csv(PREPARED_DIR / "item_ids.tsv.gz", sep="\t", index=False, compression=compression)
    manifest = {
        "schema_version": 1, "status": "awaiting_verification", "upstream_commit": carec.commit,
        "sample_fraction": SAMPLE_FRACTION, "seed": SPLIT_SEED, "sources": source_fingerprints,
        "configuration": text_config, "counts": counts, "catalogue_products": len(items),
        "sampled_review_rows": len(sampled_reviews), "sampled_metadata_products": len(sampled_metadata),
        "retained_raw_fields": {"reviews": list(text_config["inter_data_col"]), "metadata": list(text_config["item_meta_data_col"])},
        "limitations": ["Only the selected review rows and needed metadata fields are retained.",
                        "The authors' exact raw file identity and published split are not verified.",
                        "Duplicate pairs and upstream split rounding are preserved and audited."],
        "preparation_seconds": round(time.monotonic() - preparation_started, 2),
        "files": {path.name: {"bytes": path.stat().st_size, "sha256": sha256_file(path)}
                  for path in sorted(PREPARED_DIR.iterdir())},
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n")
print(f"Saved dataset: {PREPARED_DIR}")


Saved dataset: /Users/matouskovar/FIT/NI-PML/data/processed/electronics_carec_035_seed2023
